In [0]:
# Databricks notebook
# MAGIC %md
# MAGIC # 02 - Bronze: car_data e location (endpoint pesanti)
# MAGIC Scarica per sessione, per pilota e per finestre di tempo, e scrive una volta per coppia
# MAGIC (sessione, pilota). Rieseguibile: salta le coppie gia' caricate.
# MAGIC Richiede che il notebook 01 sia stato eseguito (usa `brz.sessions` e `brz.drivers`).

# COMMAND ----------

import time
import requests
from urllib.parse import urlencode, quote
from datetime import datetime, timedelta, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

dbutils.widgets.text("year", "2025")
dbutils.widgets.dropdown("endpoint", "car_data", ["car_data", "location"])
dbutils.widgets.text("window_minutes", "10")

YEAR = int(dbutils.widgets.get("year"))
CATALOG = "f1"
ENDPOINT = dbutils.widgets.get("endpoint")
WINDOW = timedelta(minutes=int(dbutils.widgets.get("window_minutes")))
BASE = "https://api.openf1.org/v1"
TABLE = f"{CATALOG}.brz.{ENDPOINT}"

# Colonne numeriche per endpoint (qui si tipizza gia' in bronze per via del volume)
INT_COLS = {
    "car_data": ["speed", "rpm", "n_gear", "throttle", "brake", "drs"],
    "location": ["x", "y", "z"],
}[ENDPOINT]
KEY_COLS = ["session_key", "meeting_key", "driver_number"]
COLS = KEY_COLS + ["date"] + INT_COLS

schema = StructType(
    [StructField(c, IntegerType(), True) for c in KEY_COLS]
    + [StructField("date", StringType(), True)]
    + [StructField(c, IntegerType(), True) for c in INT_COLS]
)


In [0]:
def get_json(path, params, raw_query=""):
    url = f"{BASE}/{path}?" + urlencode(params) + ("&" + raw_query if raw_query else "")
    for attempt in range(6):
        try:
            resp = requests.get(url, timeout=120)
            if resp.status_code == 429:
                time.sleep(2 * (attempt + 1))
                continue
            if resp.status_code == 404:
                return []
            resp.raise_for_status()
            return resp.json()
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"Richiesta fallita: {url}")


def fetch_driver_session(session_key, driver_number, start, end):
    """Scarica a finestre di tempo: filtri OpenF1 date>=... e date<... nella query string."""
    rows = []
    t = start
    while t < end:
        t2 = min(t + WINDOW, end)
        raw = ("date>=" + quote(t.isoformat(), safe=":")
               + "&date<" + quote(t2.isoformat(), safe=":"))
        rows += get_json(ENDPOINT,
                         {"session_key": session_key, "driver_number": driver_number},
                         raw_query=raw)
        t = t2
        time.sleep(0.4)
    return rows

In [0]:
sessions = spark.sql(
    f"SELECT session_key, date_start, date_end FROM {CATALOG}.brz.sessions "
    f"WHERE year = '{YEAR}' ORDER BY date_start"
).collect()

# Coppie (sessione, pilota) gia' caricate
if spark.catalog.tableExists(TABLE):
    done = {(r.session_key, r.driver_number) for r in
            spark.sql(f"SELECT DISTINCT session_key, driver_number FROM {TABLE}").collect()}
else:
    done = set()
print(f"{len(sessions)} sessioni, {len(done)} coppie gia' caricate")

In [0]:
for s in sessions:
    sk = int(s.session_key)
    if not s.date_start or not s.date_end:
        continue
    start = datetime.fromisoformat(s.date_start)
    end = datetime.fromisoformat(s.date_end)

    drivers = sorted(int(r.driver_number) for r in spark.sql(
        f"SELECT DISTINCT driver_number FROM {CATALOG}.brz.drivers "
        f"WHERE _session_key = '{sk}'").collect())

    for dn in drivers:
        if (sk, dn) in done:
            continue
        rows = fetch_driver_session(sk, dn, start, end)
        if not rows:
            print(f"session {sk} | pilota {dn}: nessun dato")
            continue
        records = [tuple(r.get(c) for c in COLS) for r in rows]
        df = (spark.createDataFrame(records, schema)
                .withColumn("date", F.to_timestamp("date"))
                .withColumn("_ingested_at", F.current_timestamp()))
        (df.write.format("delta").mode("append").saveAsTable(TABLE))
        print(f"session {sk} | pilota {dn}: {len(rows)} righe")

In [0]:
display(spark.sql(
    f"SELECT session_key, COUNT(*) AS righe, COUNT(DISTINCT driver_number) AS piloti "
    f"FROM {TABLE} GROUP BY session_key ORDER BY session_key"))